In [ ]:
import base64, os, shutil, subprocess
from openai import OpenAI

BLENDER = r"C:\Program Files\Blender Foundation\Blender 5.2\blender.exe"
OUT = "D:/Blender"
PHOTO = "D:/Blender/Diner.png"
TAG = "diner"

client = OpenAI(
    api_key=os.environ["ZAI_API_KEY"],
    base_url="https://api.z.ai/api/paas/v4/",
    timeout=900.0,
    max_retries=2,
)

def to_b64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()

In [45]:
REQS = """- Start with bpy.ops.wm.read_factory_settings(use_empty=True)
- Use scene.render.engine = 'BLENDER_EEVEE'
- Set scene.eevee.use_raytracing = True
- Set scene.eevee.taa_render_samples = 128
- Set resolution to 1280x960
- Add a camera and set scene.camera
- Render to "D:/Blender/out.png" with write_still=True
- After rendering, save with bpy.ops.wm.save_as_mainfile(filepath="D:/Blender/scene.blend")
- Keep the script under 500 lines
- Add bevel modifiers to furniture for soft edges
- Use area lights for soft shadows
- Give each material distinct roughness
- Use procedural noise for fabric, wood, and wall variation"""

In [ ]:
def _stream(prompt):
    s = client.chat.completions.create(
        model="glm-5.3-flash",
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    out = ""
    for chunk in s:
        piece = chunk.choices[0].delta.content or ""
        out += piece
        print(piece, end="")
    return out


def clean(text):
    if "```" in text:
        for p in text.split("```"):
            p = p.strip()
            if p.startswith("python"):
                return p[6:].strip()
            if p.startswith("import bpy"):
                return p
    return text.strip()


def render(script_path):
    try:
        r = subprocess.run([BLENDER, "--background", "--python", script_path],
                           capture_output=True, text=True, timeout=600)
        return r.stdout, r.stderr
    except subprocess.TimeoutExpired:
        return "", "TIMEOUT"

In [47]:
def describe_photo(path):
    resp = client.chat.completions.create(
        model="glm-5.3-flash",
        messages=[{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{to_b64(path)}"}},
            {"type": "text", "text": """Describe this room for 3D reconstruction.

Cover: room shape and dimensions in meters, wall and floor colors, window and door positions, each major object with position and size, light sources and direction.

Be concrete and spatial. No adjectives about mood."""}
        ]}],
    )
    return resp.choices[0].message.content

In [48]:
def scene_from_description(desc):
    return _stream(f"""Write a Blender 5.2 Python script that reconstructs this room:

{desc}

Requirements:
{REQS}
- Place the camera to roughly match the original photo's viewpoint
- Model furniture with real thickness and separate parts, not single boxes
- Prioritize correct layout and proportions, then material realism""")

In [49]:
def compare(photo, render_img):
    resp = client.chat.completions.create(
        model="glm-5.3-flash",
        messages=[{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{to_b64(photo)}"}},
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{to_b64(render_img)}"}},
            {"type": "text", "text": """Image 1 is a photo of a real room. Image 2 is a 3D reconstruction attempt.

List the 3 biggest differences. Focus on layout, proportions, camera angle, and object placement rather than rendering quality.

Be specific about what to change."""}
        ]}],
    )
    return resp.choices[0].message.content

In [50]:
def revise_from_photo(old_code, comparison, desc):
    return _stream(f"""Here is a Blender 5.2 Python script reconstructing a room:

{old_code}

Comparing the render to the original photo revealed these problems:

{comparison}

Rewrite the script to fix them. Pay particular attention to camera placement and object positions.

Requirements:
{REQS}""")


def repair(broken_code, error_text):
    return _stream(f"""This Blender 5.2 Python script failed:

{broken_code}

Error:
{error_text}

Fix it. Blender 5.2 uses 'BLENDER_EEVEE' as the engine name.
Output ONLY the corrected Python code, no explanation, no markdown fences.""")

In [51]:
def run_photo_loop(photo, rounds=3, tag="run"):
    desc = describe_photo(photo)
    with open(f"{OUT}/{tag}_description.txt", "w", encoding="utf-8") as f:
        f.write(desc)

    code = scene_from_description(desc)

    for i in range(rounds):
        if os.path.exists(f"{OUT}/out.png"):
            os.remove(f"{OUT}/out.png")

        with open(f"{OUT}/{tag}_scene.py", "w", encoding="utf-8") as f:
            f.write(clean(code))

        out, err = render(f"{OUT}/{tag}_scene.py")

        if not os.path.exists(f"{OUT}/out.png"):
            print(f"\nround {i}: failed, repairing")
            print(err[-500:])
            code = repair(clean(code), err[-1500:])
            continue

        shutil.copy(f"{OUT}/out.png", f"{OUT}/{tag}_{i:02d}.png")
        shutil.copy(f"{OUT}/{tag}_scene.py", f"{OUT}/{tag}_{i:02d}.py")
        if os.path.exists(f"{OUT}/scene.blend"):
            shutil.copy(f"{OUT}/scene.blend", f"{OUT}/{tag}_{i:02d}.blend")
        print(f"\nround {i}: saved")

        fixes = compare(photo, f"{OUT}/out.png")
        with open(f"{OUT}/{tag}_{i:02d}_critique.txt", "w", encoding="utf-8") as f:
            f.write(fixes)
        print("\n--- CRITIQUE ---\n", fixes)

        code = revise_from_photo(clean(code), fixes, desc)

    return code

In [52]:
final = run_photo_loop(PHOTO, rounds=5, tag=TAG)

Here's a complete, self-contained reconstruction script. It builds the full room shell (with real window/door openings), five booth units with part-built benches/tables, the counter with stools, all signage/neon, and matches the lighting description (low warm sun through the left windows, recessed downlights, pink cove, pendant). Camera sits near the front wall looking down the aisle toward the back door.

```python
# ---------------------------------------------------------------------
# Diner interior - procedural reconstruction (Blender 5.x, EEVEE + RT)
# Origin: back-left floor corner. +X = right/counter, +Y = front, +Z up.
# ---------------------------------------------------------------------
import bpy, os
from math import radians, pi
from mathutils import Vector, Matrix

# ------------------------------------------------------------- setup
bpy.ops.wm.read_factory_settings(use_empty=True)
scene = bpy.context.scene
for eng in ('BLENDER_EEVEE', 'BLENDER_EEVEE_NEXT'):
    try:
    